# EDA
Exploratory analysis of the two opportunity Excel files and the proposals/responses JSON.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA_DIR = Path('../data')
CSV_DIR  = DATA_DIR / 'csv_files'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

## 1. Load Raw Data

In [ ]:
opps1 = pd.read_excel(CSV_DIR / 'anonymized_opps_1.xlsx', sheet_name='Data')
opps2 = pd.read_excel(CSV_DIR / 'anonymized_opps_2.xlsx', sheet_name='Data')

with open(DATA_DIR / 'proposals_responses.json', encoding='utf-8') as f:
    proposals = json.load(f)

print(f'opps1 shape : {opps1.shape}')
print(f'opps2 shape : {opps2.shape}')
print(f'proposals   : {len(proposals)} entries')
print()
print(f'opps1 columns: {list(opps1.columns)}')
print(f'opps2 columns: {list(opps2.columns)}')

## 2. Clean & Normalise Column Names
Note: `opps2` has a duplicate `modified_on` column — deduplicated with a numeric suffix.

In [ ]:
def clean_cols(df):
    df = df.copy()
    cols = (
        df.columns
          .str.replace(r'\(Do Not Modify\)\s*', '', regex=True)
          .str.strip()
          .str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True)
          .str.strip('_')
    )
    # deduplicate (modified_on appears twice in opps2)
    seen = {}
    new = []
    for c in cols:
        if c in seen:
            seen[c] += 1
            new.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 1
            new.append(c)
    df.columns = new
    return df

o1 = clean_cols(opps1)
o2 = clean_cols(opps2)

print('o1 columns:', list(o1.columns))
print()
print('o2 columns:', list(o2.columns))

## 3. ID Overlap Between the Two Files

In [ ]:
ids1 = set(pd.to_numeric(o1['opportunity_id'], errors='coerce').dropna().astype(int))
ids2 = set(pd.to_numeric(o2['opportunity_id'], errors='coerce').dropna().astype(int))

print(f'IDs only in opps1      : {len(ids1 - ids2)} unique IDs')
print(f'IDs only in opps2      : {len(ids2 - ids1):>5,}')
print(f'IDs in BOTH files      : {len(ids1 & ids2):>5,}')
print(f'Total unique IDs       : {len(ids1 | ids2):>5,}')

# How many rows do those 51 opps1-exclusive IDs correspond to?
o1_only_rows = o1[o1['opportunity_id'].isin(ids1 - ids2)]
print(f'\nRows in opps1 for those 51 exclusive IDs: {len(o1_only_rows)}')
print(f'  (10 IDs appear more than once, creating 11 extra rows → 62 rows, not 51)')

# Column overlap
shared = set(o1.columns) & set(o2.columns)
only1  = set(o1.columns) - set(o2.columns)
only2  = set(o2.columns) - set(o1.columns)
print(f'\nShared columns ({len(shared)}): {sorted(shared)}')
print(f'opps1-only columns ({len(only1)}): {sorted(only1)}')
print(f'opps2-only columns ({len(only2)}): {sorted(only2)}')

## 4. Merge Strategy

**opps2** is the base (richer schema, 33 cols). For the **7,628 overlapping IDs** we left-join to bring columns exclusive to opps1 into opps2.

The **51 unique IDs exclusive to opps1** correspond to **62 rows** (10 IDs appear more than once, creating 11 extra rows); these are appended as-is and kept as duplicates because deduplication logic belongs in a downstream data-cleaning step with business rules.

In [ ]:
# Validate whether o2.total_estimated_revenue corresponds to:
# 1. o1 opportunity_estimated_revenue_base_cad
# 2. o1 service_solution_estimated_revenue
# 3. o1 opportunity_estimated_revenue_base_cad + service_solution_estimated_revenue

rev_compare = o1.copy()

rev_compare['opportunity_estimated_revenue_base_cad'] = pd.to_numeric(
    rev_compare['opportunity_estimated_revenue_base_cad'], errors='coerce'
)
rev_compare['service_solution_estimated_revenue'] = pd.to_numeric(
    rev_compare['service_solution_estimated_revenue'], errors='coerce'
)

o2_rev = o2[['opportunity_id', 'total_estimated_revenue']].copy()
o2_rev['total_estimated_revenue'] = pd.to_numeric(
    o2_rev['total_estimated_revenue'], errors='coerce'
)

# Collapse opps1 to opportunity_id level
o1_rev_by_id = (
    rev_compare
    .groupby('opportunity_id')
    .agg(
        o1_opportunity_total=('opportunity_estimated_revenue_base_cad', 'first'),
        o1_service_solution_sum=('service_solution_estimated_revenue', 'sum'),
        o1_service_solution_count=('service_solution_estimated_revenue', 'count'),
        o1_rows=('opportunity_id', 'size'),
    )
    .reset_index()
)

overlap_rev = o2_rev.merge(o1_rev_by_id, on='opportunity_id', how='inner')

overlap_rev['opportunity_plus_service'] = (
    overlap_rev['o1_opportunity_total'].fillna(0)
    + overlap_rev['o1_service_solution_sum'].fillna(0)
)

def almost_equal(a, b):
    return a.round(2).eq(b.round(2))

n = len(overlap_rev)

print(f'Overlapping opportunity IDs: {n:,}')
print()

print(
    'o2 total == o1 opportunity total:',
    f'{almost_equal(overlap_rev["total_estimated_revenue"], overlap_rev["o1_opportunity_total"]).sum():,}',
    f'/ {n:,}'
)

print(
    'o2 total == o1 service solution sum:',
    f'{almost_equal(overlap_rev["total_estimated_revenue"], overlap_rev["o1_service_solution_sum"]).sum():,}',
    f'/ {n:,}'
)

print(
    'o2 total == o1 opportunity total + service sum:',
    f'{almost_equal(overlap_rev["total_estimated_revenue"], overlap_rev["opportunity_plus_service"]).sum():,}',
    f'/ {n:,}'
)

print()
print('Median absolute difference:')
print(
    'o2 total vs o1 opportunity total:',
    (overlap_rev['total_estimated_revenue'] - overlap_rev['o1_opportunity_total']).abs().median()
)
print(
    'o2 total vs opportunity + service:',
    (overlap_rev['total_estimated_revenue'] - overlap_rev['opportunity_plus_service']).abs().median()
)

service_equals_opp = overlap_rev[
    almost_equal(
        overlap_rev['o1_service_solution_sum'],
        overlap_rev['o1_opportunity_total']
    )
]

print()
print(f'IDs where service sum equals opportunity total: {len(service_equals_opp):,}')
print(
    'Among those, o2 total equals opportunity total:',
    almost_equal(
        service_equals_opp['total_estimated_revenue'],
        service_equals_opp['o1_opportunity_total']
    ).sum()
)
print(
    'Among those, o2 total equals opportunity + service:',
    almost_equal(
        service_equals_opp['total_estimated_revenue'],
        service_equals_opp['opportunity_plus_service']
    ).sum()
)

overlap_rev[
    almost_equal(overlap_rev['total_estimated_revenue'], overlap_rev['o1_opportunity_total'])
    & ~almost_equal(overlap_rev['total_estimated_revenue'], overlap_rev['opportunity_plus_service'])
][[
    'opportunity_id',
    'total_estimated_revenue',
    'o1_opportunity_total',
    'o1_service_solution_sum',
    'opportunity_plus_service',
    'o1_service_solution_count',
    'o1_rows',
]].head(10)


In [ ]:
# Rename opps1 revenue columns to match opps2 naming.

o1_aligned = o1.rename(columns={
    'opportunity_estimated_revenue_base_cad': 'total_estimated_revenue',
    'actual_revenue_base_cad'               : 'actual_revenue',
})

# Columns present in opps1 but not opps2 — bring them in for shared rows
o1_supplement_cols = ['opportunity_id'] + [c for c in o1_aligned.columns if c not in o2.columns]
print('Supplemental columns from opps1:', o1_supplement_cols)

# Left-join the supplement onto opps2
df = o2.merge(
    o1_aligned[o1_supplement_cols].drop_duplicates(subset=['opportunity_id']),
    on='opportunity_id',
    how='left',
)

# Append rows exclusive to opps1 (51 unique IDs / 62 rows)
o1_exclusive = o1_aligned[o1_aligned['opportunity_id'].isin(ids1 - ids2)]
df = pd.concat([df, o1_exclusive], ignore_index=True)

print(f'Merged shape          : {df.shape}')
print(f'Unique opportunity_id : {df["opportunity_id"].nunique():,}')
print(f'Duplicate rows        : {df.duplicated(subset=["opportunity_id"]).sum()}')
df.head(3)

## 5. Parse Types & Inspect Missing Values

In [ ]:
date_cols = [
    'estimated_close_date', 'close_date', 'created_on',
    'proposal_submission_date', 'rfp_release_date',
    'revenue_start_date', 'current_stage_start_date',
]
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

num_cols = [
    'total_estimated_revenue', 'actual_revenue', 'probability',
    'weighted_revenue_base_cad', 'project_duration_number_of_months',
    'revenue_per_year',
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('Types set.')

In [ ]:
null_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('NULL RATES (%):')
print(null_pct[null_pct > 0].to_string())

In [ ]:
fig = px.bar(
    null_pct[null_pct > 0].reset_index(),
    x='index', y=0,
    labels={'index': 'Column', 0: '% Missing'},
    title='Missing Values by Column (%)',
    color=0,
    color_continuous_scale='reds',
    height=420
)
fig.update_xaxes(tickangle=45)
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# Raw opps1 null rates for supplemental columns
raw_o1_cols = [
    'service_solution',
    'opportunity_product',
    'ip',
    'service_solution_estimated_revenue',
    'delivery_territory_center',
]

raw_o1_nulls = (
    o1[raw_o1_cols]
    .isnull()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

print('Raw opps1 null rates (%)')
print(raw_o1_nulls.to_string())

print()
print(f'opps1 unique IDs                 : {o1["opportunity_id"].nunique():,}')
print(f'total unique IDs across files    : {len(ids1 | ids2):,}')
print(f'opps1 coverage of total unique IDs: {100 * o1["opportunity_id"].nunique() / len(ids1 | ids2):.1f}%')
print(f'overlap IDs covered by opps1      : {len(ids1 & ids2):,}')
print(f'overlap share of total unique IDs : {100 * len(ids1 & ids2) / len(ids1 | ids2):.1f}%')

**Note on null rates:** `service_solution`, `opportunity_product`, and `ip`  are 0% missing in raw opps1 (which covers ~88% of total IDs via the overlap), so the merged null rate is driven entirely by how many of those rows had matches in opps2. The columns still have limited capacity-model value because opps2 (the richer, longer-running dataset) never populated them — interpret their completeness at the opps1-origin level only.

Columns that are sparse and should be excluded from the capacity model:
- `proposal_submission_date` (~84% missing across both files)
- `rfp_release_date` (~85% missing)
- `comments` (~76% missing)
- `free_field_text_2` (~99% missing)

## 6. Status & Pipeline Stage Distribution

In [ ]:
print('=== STATUS ===')
print(df['status'].value_counts(dropna=False).to_string())
print()
print('=== SALES STAGE ===')
print(df['sales_stage'].value_counts(dropna=False).to_string())

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Status Distribution', 'Sales Stage Distribution'])

sc  = df['status'].value_counts(dropna=False)
stg = df['sales_stage'].value_counts(dropna=False)

fig.add_trace(go.Bar(x=sc.index.astype(str),  y=sc.values,  name='Status'), row=1, col=1)
fig.add_trace(go.Bar(x=stg.index.astype(str), y=stg.values, name='Stage'),  row=1, col=2)

fig.update_layout(height=380, showlegend=False, title_text='Opportunity Status & Stage')
fig.show()

In [ ]:
# Critical: how many rows are truly "open"?
open_mask = df['status'].str.lower().str.contains('open', na=False)
print(f'Open opportunities : {open_mask.sum():,} / {len(df):,} ({100*open_mask.mean():.1f}%)')
print()
print('status_reason breakdown:')
print(df['status_reason'].value_counts(dropna=False).to_string())

**Critical finding:** Only 374 open opportunities (4.3%). The dataset is overwhelmingly historical. A real-time capacity view based solely on `status = Open` will be very sparse — see section 10 for the extended load definition.

## 7. Revenue Distribution

In [ ]:
rev     = df['total_estimated_revenue'].dropna()
rev_pos = rev[rev > 0]
print(f'Revenue n={len(rev_pos):,}')
print(rev_pos.describe().apply(lambda x: f'${x:>15,.0f}'))

In [ ]:
fig = px.histogram(
    np.log10(rev_pos.clip(lower=1)),
    nbins=60,
    title='log10(Estimated Revenue) Distribution',
    labels={'value': 'log10(CAD)'},
    height=350
)
fig.show()

**Finding:** Revenue is extremely right-skewed (median $24K, max $200M). Log-scale normalization is required for the capacity score.

## 8. Probability Distribution

In [ ]:
prob = df['probability'].dropna()
n    = len(prob)

cnt_0    = (prob == 0).sum()
cnt_100  = (prob == 100).sum()
cnt_mid  = ((prob > 0) & (prob < 100)).sum()

print(f'n                  : {n:,}')
print(f'== 0               : {cnt_0:,}  ({100*cnt_0/n:.1f}%)')
print(f'== 100             : {cnt_100:,}  ({100*cnt_100/n:.1f}%)')
print(f'intermediate (1-99): {cnt_mid:,}  ({100*cnt_mid/n:.1f}%)')
print()
print(prob.describe().round(1))

fig = px.histogram(prob, nbins=20,
                   title='Probability Distribution (36.9% intermediate values)',
                   labels={'value': 'Probability (%)'},
                   height=320)
fig.show()

**Finding:** Probability is skewed heavily toward 100 (61.5%), but 36.9% of values are intermediate (1–99). It is not effectively binary. The 100-heavy skew reflects Won deals (confirmed by `status_reason`). For capacity purposes:
- Use `status` / `status_reason` as the primary won/loss signal, not probability.
- Probability retains some use as a pipeline confidence weight for open deals.

## 9. Project Duration

In [ ]:
dur = df['project_duration_number_of_months'].dropna()
dur = dur[(dur > 0) & (dur < 300)]
print(f'n={len(dur):,}')
print(dur.describe().round(1))

fig = px.histogram(dur, nbins=40,
                   title='Project Duration (months)',
                   labels={'value': 'Months'},
                   height=320)
fig.show()

**Finding:** Median 4 months; 75th pct 11 months. `revenue_start_date + project_duration` is a viable proxy for ongoing delivery commitments beyond just the open pipeline.

## 10. Director (Opportunity Manager) Analysis

In [ ]:
n_mgrs   = df['opportunity_manager'].nunique()
n_owners = df['opportunity_owner'].nunique()
same     = (df['opportunity_manager'].str.strip() == df['opportunity_owner'].str.strip()).sum()

print(f'Unique opportunity_manager : {n_mgrs}')
print(f'Unique opportunity_owner   : {n_owners}')
print(f'manager == owner           : {same:,} rows ({100*same/len(df):.1f}%)')
print('  → manager and owner are always distinct roles')

In [ ]:
print('All opportunity_owner values (likely the senior directors, n=24):')
print(df['opportunity_owner'].value_counts(dropna=False).to_string())

In [ ]:
mgr_total = df.groupby('opportunity_manager').size().sort_values(ascending=False)
print('Top 20 opportunity_manager by all-time opp count:')
print(mgr_total.head(20).to_string())

In [ ]:
fig = px.bar(
    mgr_total.head(30).reset_index(),
    x='opportunity_manager', y=0,
    title='Top 30 Managers — All-Time Opportunity Count',
    labels={'opportunity_manager': 'Manager', 0: 'Count'},
    height=400
)
fig.update_xaxes(tickangle=50)
fig.show()

In [ ]:
# Active deals and weighted pipeline per manager
# X-axis: Number of currently open opportunities assigned to that manager.
# Y-axis: Total weighted revenue for that manager’s open opportunities:
# weighted_revenue = probability / 100 * total_estimated_revenue

active = df[open_mask].copy()
active_per_mgr = active.groupby('opportunity_manager').size().sort_values(ascending=False)

df['weighted_rev_calc'] = (df['probability'].fillna(0) / 100) * df['total_estimated_revenue'].fillna(0)
weighted_pipeline = (
    df[open_mask]
    .groupby('opportunity_manager')['weighted_rev_calc']
    .sum()
    .sort_values(ascending=False)
)

mgr_stats = pd.DataFrame({
    'active_deals'      : active_per_mgr,
    'weighted_pipeline' : weighted_pipeline,
}).dropna()

fig = px.scatter(
    mgr_stats.reset_index(),
    x='active_deals', y='weighted_pipeline',
    text='opportunity_manager',
    title='Manager — Open Deal Count vs Weighted Pipeline',
    labels={'active_deals': '# Open Deals', 'weighted_pipeline': 'Weighted Pipeline (CAD)'},
    height=500
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.show()

**Finding:** Manager and Owner are always distinct (0% overlap). 68 managers vs. 24 owners — owners are likely the senior directors. Dashboard primary unit = owner; manager breakdown as drill-down.

## 11. Historical Baseline — Quarterly Concurrent Deal Load

In [ ]:
# Derive range from the data itself so new records are always included
data_start = df['created_on'].dropna().min()
data_end   = df['created_on'].dropna().max()

q_start_period = pd.Period(data_start, freq='Q')
q_end_period   = pd.Period(data_end,   freq='Q')

print(f'Data runs: {data_start.date()} → {data_end.date()}')
print(f'Quarter range: {q_start_period} → {q_end_period}')

quarters = pd.period_range(q_start_period, q_end_period, freq='Q')

In [ ]:
rows = []
for q in quarters:
    qs = q.start_time
    qe = q.end_time
    mask = (
        df['created_on'].notna() &
        (df['created_on'] <= qe) &
        (df['close_date'].isna() | (df['close_date'] >= qs))
    )
    per_mgr = df[mask].groupby('opportunity_manager').size().reset_index(name='deal_count')
    per_mgr['quarter'] = str(q)
    rows.append(per_mgr)

timeline = pd.concat(rows, ignore_index=True)
print(f'Timeline rows: {len(timeline):,}  covering {len(quarters)} quarters')

In [ ]:
baseline = timeline.groupby('opportunity_manager')['deal_count'].agg(
    hist_mean='mean',
    hist_std='std',
    hist_max='max',
    quarters_seen='count'
).round(2).sort_values('hist_mean', ascending=False)

print(f'Managers with <4 quarters history (unreliable baseline): {(baseline["quarters_seen"] < 4).sum()}')
print(f'Managers with ≥4 quarters history: {(baseline["quarters_seen"] >= 4).sum()}')
print()
print('Top 15 managers by historical mean concurrent deal count:')
print(baseline.head(15).to_string())

In [ ]:
top5 = mgr_total.head(5).index.tolist()

fig = px.line(
    timeline[timeline['opportunity_manager'].isin(top5)],
    x='quarter', y='deal_count',
    color='opportunity_manager',
    title='Concurrent Deal Count Over Time — Top 5 Managers (full date range)',
    labels={'deal_count': 'Concurrent Deals', 'quarter': 'Quarter'},
    height=420
)
fig.show()

## 12. Territory, Opportunity Type & Sales Model

In [ ]:
print('=== PRIMARY TERRITORY ===')
print(df['primary_territory'].value_counts(dropna=False).to_string())

print('\n=== OPPORTUNITY TYPE ===')
print(df['opportunity_type'].value_counts(dropna=False).to_string())

print('\n=== SALES MODEL ===')
print(df['sales_model'].value_counts(dropna=False).to_string())

**Finding:** Data is scoped to Atlantic Canada only — not pan-Canada as the problem statement describes. Clarify scope with CGI before final delivery.

## 13. Proposals / Responses JSON

In [ ]:
def approx_tokens(lines):
    """Word-count estimate only (~4/3 words→tokens). Use tiktoken before
    finalising chunk counts against the actual cl100k_base tokenizer."""
    return len(' '.join(lines).split()) * 4 // 3

rows_p = []
for title, entry in proposals.items():
    p = entry['proposal']
    prop_toks = approx_tokens(p.get('content', []))
    resp_toks = sum(
        approx_tokens(v.get('content', []))
        for v in p.get('proposal_response', {}).values()
    )
    rows_p.append({
        'title'          : title[:65],
        'proposal_tokens': prop_toks,
        'response_tokens': resp_toks,
        'total_tokens'   : prop_toks + resp_toks,
    })

pf = pd.DataFrame(rows_p).sort_values('proposal_tokens', ascending=False)
pf

In [ ]:
# OpenAI embeddings API limit is 8192 tokens per input
# https://platform.openai.com/docs/api-reference/embeddings/create
EMB_LIMIT = 8_192

need_chunk = pf[pf['proposal_tokens'] > EMB_LIMIT]
print(f'Embedding limit (text-embedding-3-small) : {EMB_LIMIT:,} tokens')
print(f'Proposals exceeding limit (word estimate): {len(need_chunk)} / {len(pf)}')
print()
print('NOTE: approx_tokens() uses a word-count heuristic.')
print('Verify with tiktoken (cl100k_base) before finalising chunk sizes.')
print()
print(need_chunk[['title', 'proposal_tokens']].to_string(index=False))

In [ ]:
fig = px.bar(
    pf,
    x='title', y=['proposal_tokens', 'response_tokens'],
    barmode='stack',
    title='Approx Token Length per Proposal (word-count estimate)',
    labels={'value': 'Approx Tokens', 'title': ''},
    height=430
)
fig.add_hline(y=EMB_LIMIT, line_dash='dash', line_color='red',
              annotation_text='8192 embedding limit')
fig.update_xaxes(tickangle=50)
fig.show()

**Finding:** 15 of 19 proposals exceed the 8,192-token embedding limit (word-count estimate). Chunking with overlap is mandatory. Token counts should be verified with `tiktoken` (cl100k_base tokenizer) before finalizing chunk sizes, as the word-count heuristic can be off by ±20%.

## 14. EDA Summary

In [ ]:
print('=' * 65)
print('DATA SHAPE')
print('=' * 65)
print(f'  Merged rows                     : {len(df):,}')
print(f'  Unique opportunity IDs          : {df["opportunity_id"].nunique():,}')
print(f'  Duplicate rows (opps1 multi-row): {df.duplicated(subset=["opportunity_id"]).sum()}')
print(f'  Open (active) opportunities     : {open_mask.sum():,} ({100*open_mask.mean():.1f}%)')
print(f'  Unique opportunity_manager      : {df["opportunity_manager"].nunique()}')
print(f'  Unique opportunity_owner        : {df["opportunity_owner"].nunique()}')

print()
print('=' * 65)
print('GENUINELY SPARSE COLUMNS (drop from capacity model)')
print('=' * 65)
genuinely_sparse = [
    'proposal_submission_date', 'rfp_release_date',
    'comments', 'free_field_text_2', 'is_dpsc_required_updated_on'
]
for c in genuinely_sparse:
    if c in df.columns:
        print(f'  {c:<45}: {null_pct.get(c, 0):.1f}% missing')

print()
print('=' * 65)
print('PROBABILITY')
print('=' * 65)
print(f'  Skewed high (61.5% == 100), but 36.9% intermediate')
print(f'  Use status/status_reason for win/loss; probability')
print(f'  retains value as pipeline confidence weight for open deals')

print()
print('=' * 65)
print('PROPOSALS')
print('=' * 65)
print(f'  {len(pf)} total  |  {len(need_chunk)} exceed 8192-token limit (word estimate)')
print(f'  Largest: {pf["proposal_tokens"].max():,} tokens — verify with tiktoken')

print()
print('=' * 65)
print('SCOPE NOTE')
print('=' * 65)
print(f'  Geography: Atlantic Canada only')
print(f'  Date range: {data_start.date()} → {data_end.date()}')
print(f'  Manager vs Owner: always distinct (68 managers / 24 owners)')